# 实验03：正则表达式与网络结构化数据
学号：20244080119
姓名：李奕昕
环境：py_exp

### 任务 0：环境记录
记录当前 Conda 环境、学号、机器指纹、RUN_ID 和 Python 版本。

In [47]:
import hashlib
import os
import platform
import sys
import uuid

STUDENT_ID = 20244080119
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "未检测到 CONDA_DEFAULT_ENV")
machine_info = (
    f"{platform.node()}|{platform.machine()}|"
    f"{platform.system()}|{uuid.getnode()}"
)
MACHINE_FP = hashlib.sha256(machine_info.encode()).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"

print(f"[Conda 环境] {CONDA_ENV}")
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")
print(f"[Python 版本] {platform.python_version()}")
print(f"[Python 解释器] {sys.executable}")

[Conda 环境] py_exp
[学号] 20244080119
[机器指纹] 60bb97c2c4e89b33
[RUN_ID] 20244080119-60bb97c2c4e89b33
[Python 版本] 3.11.17
[Python 解释器] d:\Anaconda\envs\py_exp\python.exe


### 任务 1.1：匹配数字、空白和完整单词
在给定文本中分别匹配数字、空白字符和完整的 cat 单词。\bcat\b 不会匹配 cats 中的 cat。

In [48]:
import re

text = "cat on mat; later: cats and dogs"
digit_matches = re.findall(r"\d+", text)
whitespace_matches = re.findall(r"\s+", text)
whole_cat_matches = re.findall(r"\bcat\b", text)

print(f"原文: {text!r}")
print(f"数字 \\d+: {digit_matches}")
print(f"空白 \\s+: {whitespace_matches}")
print(f"完整单词 \\bcat\\b: {whole_cat_matches}")

原文: 'cat on mat; later: cats and dogs'
数字 \d+: []
空白 \s+: [' ', ' ', ' ', ' ', ' ', ' ']
完整单词 \bcat\b: ['cat']


### 任务 1.2：测试选择分支及匹配顺序
cat|dog 可以匹配两种单词。正则表达式按从左到右的顺序尝试分支，因此 a|aa|aaa 与 aaa|aa|a 会产生不同结果。

In [49]:
import re

text = "cat dog bird CAT"
alternation_matches = re.findall(r"cat|dog", text)
short_first = re.findall(r"a|aa|aaa", "aaaa")
long_first = re.findall(r"aaa|aa|a", "aaaa")

print(f"原文: {text!r}")
print(f"cat|dog 的匹配结果: {alternation_matches}")
print(f"a|aa|aaa 的匹配结果: {short_first}")
print(f"aaa|aa|a 的匹配结果: {long_first}")

原文: 'cat dog bird CAT'
cat|dog 的匹配结果: ['cat', 'dog']
a|aa|aaa 的匹配结果: ['a', 'a', 'a', 'a']
aaa|aa|a 的匹配结果: ['aaa', 'a']


### 任务 1.3：比较重复次数限定符
分别测试 *、+、?、{2} 和 {1,2} 对字符 a 重复次数的限制。

In [50]:
import re

text = "ct cat caat caaat"
patterns = [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]

print(f"原文: {text!r}")
for pattern in patterns:
    print(f"{pattern}: {re.findall(pattern, text)}")

原文: 'ct cat caat caaat'
ca*t: ['ct', 'cat', 'caat', 'caaat']
ca+t: ['cat', 'caat', 'caaat']
ca?t: ['ct', 'cat']
ca{2}t: ['caat']
ca{1,2}t: ['cat', 'caat']


### 任务 2.1：提取邮箱用户名与域名
使用捕获组分别提取邮箱地址中的用户名和域名。

In [51]:
import re

text = "联系: alice@example.com; 备选: bob@test.org"
pattern = r"([\w.+-]+)@([\w.-]+\.[A-Za-z]{2,})"

print(f"原文: {text}")
for match in re.finditer(pattern, text):
    username, domain = match.groups()
    print(f"邮箱: {match.group(0)} -> 用户名: {username}, 域名: {domain}")

原文: 联系: alice@example.com; 备选: bob@test.org
邮箱: alice@example.com -> 用户名: alice, 域名: example.com
邮箱: bob@test.org -> 用户名: bob, 域名: test.org


### 任务 2.2：查找连续重复的单词
使用回溯引用 \1 查找相邻且相同的单词，忽略大小写。

In [52]:
import re

text = "cat cat, dog cat, bird bird"
repeated_words = re.findall(
    r"\b(\w+)\s+\1\b",
    text,
    flags=re.IGNORECASE,
)

print(f"原文: {text}")
print(f"连续重复的单词: {repeated_words}")

原文: cat cat, dog cat, bird bird
连续重复的单词: ['cat', 'bird']


### 任务 2.3：将日期从“年-月-日”改为“月-日-年”
使用捕获组取出年月日，再按月、日、年的顺序替换。替换时复用第 2 个捕获组作为分隔符，因此连字符、斜杠和句点都会保留。逐行处理以保留原有行数。

In [53]:
import re

dates = "2024-05-01\n2024/05/02\n2024.05.03"
date_pattern = re.compile(r"(\d{4})([-/.])(\d{2})\2(\d{2})")
converted_dates = date_pattern.sub(r"\3\2\4\2\1", dates)

print("转换前:")
print(dates)
print("\n转换后（月-日-年）:")
print(converted_dates)
print(f"\n转换前行数: {len(dates.splitlines())}")
print(f"转换后行数: {len(converted_dates.splitlines())}")

转换前:
2024-05-01
2024/05/02
2024.05.03

转换后（月-日-年）:
05-01-2024
05/02/2024
05.03.2024

转换前行数: 3
转换后行数: 3


### 任务 2.4：统一电话号码格式
将连字符或点分隔的电话号码统一为（区号）号码前半段-号码后半段格式。

In [54]:
import re

text = "联系我: 123-456-7890 或 987.654.3210"
phone_pattern = re.compile(r"(?<!\d)(\d{3})[-.](\d{3})[-.](\d{4})(?!\d)")
normalized_text = phone_pattern.sub(
    lambda m: f"({m.group(1)}) {m.group(2)}-{m.group(3)}",
    text,
)

print(f"转换前: {text}")
print(f"转换后: {normalized_text}")
print(f"匹配到的电话号码数: {len(phone_pattern.findall(text))}")

转换前: 联系我: 123-456-7890 或 987.654.3210
转换后: 联系我: (123) 456-7890 或 (987) 654-3210
匹配到的电话号码数: 2


### 任务 3.1：创建解析树并提取标题和 h1
使用题目给定的 HTML 文档创建解析树，并提取标题和 h1 标签信息。

In [55]:
from bs4 import BeautifulSoup

html_doc = """<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲: 正则表达式</p>
<p class="course">第七讲: 结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>"""

soup = BeautifulSoup(html_doc, "html.parser")
title = soup.title
heading = soup.find("h1")

print(f"页面标题文本: {title.get_text()}")
print(f"h1 标签名: {heading.name}")
print(f"h1 文本: {heading.get_text()}")

页面标题文本: 实验课程
h1 标签名: h1
h1 文本: 数据分析


### 任务 3.2：查找课程段落和所有链接
使用 find_all() 查找 class="course" 的段落和所有链接，逐项打印文字与 href。本单元使用任务 3.1 中定义的解析树 soup。

In [56]:
course_paragraphs = soup.find_all("p", class_="course")
all_links = soup.find_all("a")

print(f"课程段落数量: {len(course_paragraphs)}")
for i, paragraph in enumerate(course_paragraphs, 1):
    print(f"课程段落 {i}: {paragraph.get_text()}")

print(f"\n链接数量: {len(all_links)}")
for i, link in enumerate(all_links, 1):
    print(f"链接 {i}: 文字={link.get_text()!r}, href={link['href']!r}")

课程段落数量: 2
课程段落 1: 第六讲: 正则表达式
课程段落 2: 第七讲: 结构化数据

链接数量: 2
链接 1: 文字='讲义六', href='https://example.org/lecture06'
链接 2: 文字='讲义七', href='https://example.org/lecture07'


### 任务 3.3：比较不同范围的 get_text() 结果
分别提取整篇文档、单个段落和单个链接的文本。调用对象不同，提取的文本范围也不同。

In [57]:
full_text = soup.get_text()
paragraph_text = course_paragraphs[0].get_text()
link_text = all_links[0].get_text()

print("整个文档 get_text():")
print(repr(full_text))
print(f"\n单个段落 get_text(): {paragraph_text!r}")
print(f"单个链接 get_text(): {link_text!r}")

整个文档 get_text():
'实验课程\n数据分析\n第六讲: 正则表达式\n第七讲: 结构化数据\n讲义六\n讲义七\n'

单个段落 get_text(): '第六讲: 正则表达式'
单个链接 get_text(): '讲义六'
